# Notebook 11 — Fraud-Risk Signal Integration and Ranking Diagnostics

**Purpose:** Compare the existing supervised diversion-disclosure model with the revised Notebook 10 financial/governance anomaly indicators; examine redundancy, ranking overlap, and a transparent **exploratory** consensus queue for criminal-investigator review.

**Scope:** 2024 full Form 990 filings only. The 990-EZ / unknown-label population is **not** included. The supervised outcome is *self-reported significant diversion of assets*, **not confirmed fraud**. Anomaly indicators are heuristics, not evidence of misconduct.

**Important:** This notebook does **not** train or tune a new fraud model, use 2024 labels to choose weights, or replace the Notebook 09/10 models. It uses the saved Notebook 09 model to score the eligible cohort and uses Notebook 10's existing unsupervised outputs. The 2023–24 period has already been examined and is **not an untouched holdout**.

**Outputs:** `data/processed/notebook11/`. Earlier notebook outputs are read-only. Run from the repository root or `notebooks/` in your `irs990` environment.

## 1. Configuration and input checks

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import duckdb
import joblib
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data/processed'
FEATURES_PATH = DATA / 'nccs_longitudinal_features_v1.parquet'
QUEUE10 = DATA / 'notebook10/06_2024_form990_anomaly_queue.parquet'
MODEL09 = DATA / 'notebook09/selected_model.joblib'
MANIFEST09 = DATA / 'notebook09/model_manifest.json'
OUT = DATA / 'notebook11'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
TOP_K = (100, 500, 1000)
# A deliberately equal-weight exploratory consensus; NOT an estimated fraud probability.
CONSENSUS_WEIGHTS = {'supervised_percentile': 0.5, 'peer_percentile': 0.5}
for p in (FEATURES_PATH, QUEUE10, MODEL09, MANIFEST09):
    assert p.is_file(), f'Missing prerequisite: {p}'
manifest = json.loads(MANIFEST09.read_text())
MODEL_FEATURES = manifest['features']
assert 'fraud_label' not in MODEL_FEATURES

def sqlpath(path):
    return str(path).replace("'", "''")

con = duckdb.connect()
con.execute('SET threads=4')
con.execute("SET memory_limit='6GB'")
con.execute(f"CREATE VIEW features AS SELECT * FROM read_parquet('{sqlpath(FEATURES_PATH)}')")
con.execute(f"CREATE VIEW anomaly AS SELECT * FROM read_parquet('{sqlpath(QUEUE10)}')")
print('Project:', ROOT)
print('Model:', manifest.get('selected_model', 'not specified'))
print('Notebook 11 output:', OUT)
print('Required model features:', len(MODEL_FEATURES))

## 2. Join integrity and cohort scope

The join is on the selected filing identifier (`OBJECTID`), not EIN alone. Assert a one-to-one match to avoid accidentally mixing filings across tax years. All Notebook 10 rows must be full Form 990, 2024, with a known disclosure label.

In [ ]:
source_cols = set(con.execute('DESCRIBE features').df().column_name)
queue_cols = set(con.execute('DESCRIBE anomaly').df().column_name)
need_queue = {'OBJECTID','ein','tax_year','review_cohort','available_indicator_count',
              'adverse_peer_index','peer_anomaly_index','isolation_anomaly_score',
              'primary_rank','expense_revenue_ratio','net_margin'}
assert set(MODEL_FEATURES).issubset(source_cols), sorted(set(MODEL_FEATURES)-source_cols)
assert need_queue.issubset(queue_cols), sorted(need_queue-queue_cols)
join_check = con.execute('''
SELECT (SELECT COUNT(*) FROM anomaly) queue_rows,
       (SELECT COUNT(DISTINCT OBJECTID) FROM anomaly) queue_unique_ids,
       COUNT(*) joined_rows,
       COUNT(*) FILTER (WHERE f.OBJECTID IS NULL) unmatched,
       COUNT(*) FILTER (WHERE f.fraud_label NOT IN (0,1) OR f.fraud_label IS NULL) unknown_labels,
       COUNT(*) FILTER (WHERE q.tax_year<>2024 OR f.tax_year<>2024) wrong_year,
       COUNT(*) FILTER (WHERE CAST(q.ein AS VARCHAR)<>CAST(f.ein AS VARCHAR)) ein_mismatch
FROM anomaly q LEFT JOIN features f ON q.OBJECTID=f.OBJECTID AND f.tax_year=2024
''').df()
display(join_check)
r = join_check.iloc[0]
assert r.queue_rows == r.queue_unique_ids == r.joined_rows
assert r.unmatched == r.unknown_labels == r.wrong_year == r.ein_mismatch == 0
assert con.execute("SELECT COUNT(*) FROM anomaly WHERE review_cohort NOT IN ('primary_review','limited_data') OR review_cohort IS NULL").fetchone()[0] == 0
join_check.to_csv(OUT/'01_join_integrity.csv', index=False)

## 3. Score the 2024 Form 990 cohort with the saved Notebook 09 model

Load only the 18 predictors specified by the saved model manifest. Do not retrain or refit the model. Scores are uncalibrated and should not be interpreted as probabilities of fraud. The `joblib` artifact should only be loaded if it was created locally and is trusted.

In [ ]:
# The saved Notebook 09 model is a trusted local artifact from this project.
model = joblib.load(MODEL09)
quoted = ', '.join('f."'+c.replace('"','""')+'"' for c in MODEL_FEATURES)
raw = con.execute(f'''
    SELECT q.OBJECTID, f.fraud_label, {quoted}
    FROM anomaly q JOIN features f ON q.OBJECTID=f.OBJECTID AND f.tax_year=2024
    ORDER BY q.OBJECTID
''').df()
X = raw[MODEL_FEATURES].apply(pd.to_numeric, errors='coerce').replace([np.inf,-np.inf],np.nan)
assert len(X) == len(raw)
score = np.empty(len(X), dtype=np.float64)
for start in range(0, len(X), 50_000):
    end = min(start+50_000, len(X))
    score[start:end] = model.predict_proba(X.iloc[start:end])[:,1]
assert np.isfinite(score).all()
supervised = pd.DataFrame({'OBJECTID':raw.OBJECTID, 'fraud_label':raw.fraud_label,
                           'supervised_diversion_score':score})
assert supervised.OBJECTID.is_unique
supervised.to_parquet(OUT/'02_2024_supervised_scores.parquet', index=False)
print('Scored filings:', len(supervised))
display(supervised.supervised_diversion_score.describe(percentiles=[.5,.9,.99,.999]).to_frame())
del raw, X, model

## 4. Detect correlated / potentially double-counted indicators

Expense-to-revenue ratio and net margin may be mathematically related. Examine their correlations in the 2024 cohort, along with other anomaly inputs. **Do not treat a high correlation as proof of fraud or use disclosure labels to select features.**

For the exploratory consensus, construct a *six-indicator peer score* that excludes `net_margin` to reduce direct redundancy with expense/revenue. Preserve the original seven-indicator score for comparison. The six-indicator score is a **heuristic**, not a validated replacement for Notebook 10.

In [ ]:
indicators = ['expense_revenue_ratio','liabilities_assets_ratio','net_margin',
              'revenue_growth_3yr','asset_growth_3yr','revenue_volatility_3yr','board_independence']
allcols = set(con.execute('DESCRIBE anomaly').df().column_name)
assert set(indicators).issubset(allcols)
assert all('adverse_'+c in allcols for c in indicators)
indicator_df = con.execute('SELECT '+', '.join(indicators)+' FROM anomaly').df()
indicator_df = indicator_df.replace([np.inf,-np.inf],np.nan)
corr = indicator_df.corr(method='spearman', min_periods=100)
display(corr.round(3))
corr.to_csv(OUT/'03_indicator_spearman_correlations.csv')
# The six-indicator score deliberately drops net_margin, but retains its
# underlying measurements for analyst context. Missing components count as zero.
SIX = [c for c in indicators if c != 'net_margin']
print('Original indicators:', len(indicators), '| de-duplicated indicators:', len(SIX))
print('Expense/revenue vs net margin Spearman correlation:',
      round(float(corr.loc['expense_revenue_ratio','net_margin']), 4))
del indicator_df

## 5. Construct transparent comparison rankings

**Three distinct analytical signals:** supervised diversion-disclosure score; directional peer score (six-indicator version); and Isolation Forest score (retained for comparison, **not** included in the consensus because it reuses many of the same underlying variables).

Within the **primary-review** cohort, rank the supervised and peer scores and convert ranks to percentile positions. The exploratory consensus is the mean of these two percentile positions, with fixed 50/50 weights chosen for demonstration, **not fitted to the 2024 labels**. The limited-data cohort remains separate and is never assigned a primary rank.

**Interpretation:** Consensus means *agreement between two imperfect screening methods*, not calibrated fraud likelihood.

In [ ]:
q = con.execute('SELECT * FROM anomaly ORDER BY OBJECTID').df()
q = q.merge(supervised[['OBJECTID','supervised_diversion_score','fraud_label']], on='OBJECTID', how='left', validate='one_to_one')
assert q.supervised_diversion_score.notna().all()
components = ['adverse_'+c for c in SIX]
q['six_indicator_count'] = q[components].notna().sum(axis=1).astype('int8')
q['peer_adverse_6'] = q[components].fillna(0).sum(axis=1)/len(SIX)
assert q.peer_adverse_6.between(0,1).all()
# Preserve Notebook 10's original eligibility policy; also require 4 of 6
# observed indicators for the *new* consensus ranking.
q['integration_cohort'] = np.where(
    q.review_cohort.eq('primary_review') & q.six_indicator_count.ge(4),
    'primary_review', 'limited_data')
primary = q.integration_cohort.eq('primary_review')
for name,col in [('supervised','supervised_diversion_score'),
                 ('peer','peer_adverse_6'),('peer_original','adverse_peer_index'),
                 ('isolation','isolation_anomaly_score')]:
    q[name+'_rank'] = np.nan
    valid = primary & q[col].notna()
    q.loc[valid,name+'_rank'] = q.loc[valid,col].rank(method='first',ascending=False)
    q[name+'_percentile'] = np.nan
    # High percentile = high rank / more anomalous, not fraud probability.
    nvalid = int(valid.sum())
    if nvalid:
        q.loc[valid,name+'_percentile'] = 1 - (q.loc[valid,name+'_rank']-1)/nvalid
q['consensus_score'] = np.nan
q.loc[primary,'consensus_score'] = (
    CONSENSUS_WEIGHTS['supervised_percentile']*q.loc[primary,'supervised_percentile'] +
    CONSENSUS_WEIGHTS['peer_percentile']*q.loc[primary,'peer_percentile'])
q['consensus_rank'] = np.nan
q.loc[primary,'consensus_rank'] = q.loc[primary,'consensus_score'].rank(method='first',ascending=False)
q['consensus_rank'] = q['consensus_rank'].astype('Int64')
assert q.OBJECTID.is_unique
assert q.loc[primary,'consensus_rank'].is_unique
assert q.loc[primary,'consensus_rank'].min()==1
assert q.loc[primary,'consensus_rank'].max()==primary.sum()
assert q.loc[~primary,'consensus_rank'].isna().all()
assert q.loc[primary,'consensus_score'].between(0,1).all()
print('Integrated primary cohort:', int(primary.sum()))
print('Limited-data cohort:', int((~primary).sum()))
display(q.groupby('integration_cohort',dropna=False).agg(filings=('OBJECTID','size'),
        mean_six_indicators=('six_indicator_count','mean'),
        mean_consensus=('consensus_score','mean')).reset_index())

## 6. Rank overlap, agreement, and double-counting diagnostics

Compare the top-K filings across the two component rankings and the consensus. Large disagreement means the methods surface different organizations; agreement does **not** establish misconduct. Quantify the effect of dropping `net_margin` from the peer score.

In [ ]:
from scipy.stats import spearmanr
p = q.loc[primary].copy()
rows=[]
for k in TOP_K:
    sets={name:set(p.nsmallest(k,col).OBJECTID) for name,col in [
        ('supervised','supervised_rank'),('peer_6','peer_rank'),
        ('peer_7','peer_original_rank'),('isolation','isolation_rank'),
        ('consensus','consensus_rank')] if p[col].notna().sum()>=k}
    row={'top_k':k}
    for a,b in [('supervised','peer_6'),('supervised','consensus'),
                ('peer_6','consensus'),('peer_6','peer_7'),
                ('isolation','consensus')]:
        row[f'{a}_vs_{b}_overlap'] = len(sets[a]&sets[b]) if a in sets and b in sets else np.nan
    rows.append(row)
overlap = pd.DataFrame(rows)
display(overlap)
overlap.to_csv(OUT/'04_topk_overlap.csv',index=False)
rankcols=['supervised_percentile','peer_percentile','peer_original_percentile',
          'isolation_percentile','consensus_score']
rank_corr=p[rankcols].corr(method='spearman')
display(rank_corr.round(3))
rank_corr.to_csv(OUT/'04_rank_spearman_correlations.csv')
# Investigate the effect of a second weighting assumption, without selecting
# a winner based on 2024 labels.
for w in (0.25,0.75):
    col=f'consensus_supervised_weight_{int(w*100)}'
    p[col]=w*p.supervised_percentile+(1-w)*p.peer_percentile
    p[col+'_rank']=p[col].rank(ascending=False,method='first')
weight_rows=[]
for k in TOP_K:
    reference=set(p.nsmallest(k,'consensus_rank').OBJECTID)
    for w in (25,75):
        alt=set(p.nsmallest(k,f'consensus_supervised_weight_{w}_rank').OBJECTID)
        weight_rows.append({'top_k':k,'supervised_weight':w/100,
                            'overlap_with_equal_weight':len(reference&alt)})
weight_check=pd.DataFrame(weight_rows)
display(weight_check)
weight_check.to_csv(OUT/'04_weight_sensitivity.csv',index=False)

## 7. Retrospective disclosure overlap (descriptive only)

The target is **self-reported significant diversion**, not independently verified fraud. This is an already-examined 2024 period. Do not use these results to optimize weights, declare model superiority, or make prospective accuracy claims. The anomaly model does not use the disclosure label; the supervised model was trained on earlier disclosures.

In [ ]:
diagnostics=[]
for name,col in [('supervised','supervised_rank'),('peer_6','peer_rank'),
                 ('peer_7','peer_original_rank'),('isolation','isolation_rank'),
                 ('consensus','consensus_rank')]:
    valid=p[p[col].notna()].copy()
    for k in TOP_K:
        top=valid.nsmallest(k,col)
        diagnostics.append({'signal':name,'top_k':k,'filings':len(top),
                            'reported_diversions':int(top.fraud_label.eq(1).sum()),
                            'positive_rate':float(top.fraud_label.eq(1).mean()) if len(top) else np.nan})
label_diag=pd.DataFrame(diagnostics)
display(label_diag.pivot(index='signal',columns='top_k',values='reported_diversions'))
print('2024 primary-review positive disclosures:', int(p.fraud_label.eq(1).sum()))
label_diag.to_csv(OUT/'05_retrospective_disclosure_overlap.csv',index=False)

## 8. Export investigator-oriented research queue

This is a **research prioritization artifact** only. It includes the separate component scores, ranks, coverage, top adverse indicators, and explicit caveats. Do not label the composite as `fraud_probability`, and do not infer wrongdoing from its rank. All limited-data filings are retained in a separate file. No prior notebook artifacts are overwritten.

In [ ]:
# Each indicator's adverse score is a relative screening flag, not an allegation.
contributions=['adverse_'+c for c in SIX]
pretty={
    'expense_revenue_ratio':'High expenses relative to revenue',
    'liabilities_assets_ratio':'High liabilities relative to assets',
    'revenue_growth_3yr':'Low three-year revenue growth',
    'asset_growth_3yr':'Low three-year asset growth',
    'revenue_volatility_3yr':'High revenue volatility',
    'board_independence':'Low board independence',
}
arr=q[contributions].fillna(-1).to_numpy(dtype=float)
maxidx=np.argmax(arr,axis=1)
maxval=np.max(arr,axis=1)
q['leading_peer_indicator']=[pretty[SIX[i]] if v>0 else 'No adverse peer indicator' for i,v in zip(maxidx,maxval)]
q['leading_peer_indicator_severity']=np.maximum(maxval,0)
q['interpretation_warning']='Statistical screening only; not evidence of fraud or a calibrated fraud probability.'
q['target_warning']='Supervised score models self-reported diversion, not confirmed or undetected fraud.'
# Keep identifiers, scores, coverage, contextual metrics, and individual components.
columns=['OBJECTID','ein','tax_year','peer_group','integration_cohort',
         'review_cohort','available_indicator_count','six_indicator_count',
         'revenue','expenses','assets','liabilities','net_income',
         'supervised_diversion_score','supervised_rank','supervised_percentile',
         'peer_adverse_6','peer_rank','peer_percentile',
         'adverse_peer_index','peer_original_rank','isolation_anomaly_score',
         'isolation_rank','consensus_score','consensus_rank',
         'leading_peer_indicator','leading_peer_indicator_severity']+contributions+[
         'interpretation_warning','target_warning']
assert set(columns).issubset(q.columns),sorted(set(columns)-set(q.columns))
primary_queue=q.loc[primary,columns].sort_values('consensus_rank')
limited_queue=q.loc[~primary,columns].sort_values(['six_indicator_count','OBJECTID'],ascending=[False,True])
primary_queue.to_parquet(OUT/'06_2024_integrated_primary_queue.parquet',index=False)
limited_queue.to_parquet(OUT/'06_2024_limited_data_queue.parquet',index=False)
primary_queue.head(1000).to_csv(OUT/'06_top1000_research_leads.csv',index=False)
display(primary_queue[['consensus_rank','ein','peer_group','six_indicator_count',
                       'supervised_rank','peer_rank','consensus_score',
                       'leading_peer_indicator']].head(20))
print('Saved primary:',len(primary_queue),'limited:',len(limited_queue))

## 9. Final QA, manifest, and interpretation

Review coverage, missing scores, duplicate identifiers, rank continuity, and output counts. If the source data or model changes, rerun this notebook and document the change. The consensus is a **transparent comparison tool**, not a validated criminal-investigation risk model.

In [ ]:
qa={
    'source_queue_rows':int(len(q)),
    'primary_rows':int(len(primary_queue)),
    'limited_rows':int(len(limited_queue)),
    'unique_OBJECTIDs':int(q.OBJECTID.nunique()),
    'supervised_missing':int(q.supervised_diversion_score.isna().sum()),
    'primary_consensus_missing':int(primary_queue.consensus_score.isna().sum()),
    'limited_with_primary_rank':int(limited_queue.consensus_rank.notna().sum()),
    'primary_min_six_indicators':int(primary_queue.six_indicator_count.min()),
    'primary_positive_disclosures':int(p.fraud_label.eq(1).sum()),
}
assert qa['source_queue_rows']==qa['primary_rows']+qa['limited_rows']==qa['unique_OBJECTIDs']
assert qa['supervised_missing']==qa['primary_consensus_missing']==qa['limited_with_primary_rank']==0
assert qa['primary_min_six_indicators']>=4
assert sorted(primary_queue.consensus_rank.astype(int).tolist())==list(range(1,len(primary_queue)+1))
display(pd.DataFrame([qa]))
(OUT/'07_quality_checks.json').write_text(json.dumps(qa,indent=2))
metadata={
    'notebook':'11 — Fraud-Risk Signal Integration and Ranking Diagnostics',
    'year':2024,
    'population':'Full Form 990, eligible 09D/10 cohort; 990-EZ excluded',
    'source_feature_file':str(FEATURES_PATH),
    'source_anomaly_queue':str(QUEUE10),
    'supervised_model':str(MODEL09),
    'supervised_model_manifest':str(MANIFEST09),
    'supervised_target':'Self-reported significant diversion of assets, not verified fraud',
    'supervised_model_name':manifest.get('selected_model'),
    'peer_score':'Six directional indicators; net_margin excluded to reduce redundancy with expense/revenue; missing contributions=0',
    'primary_eligibility':'Notebook 10 primary_review and at least 4 of 6 remaining indicators',
    'consensus':'Fixed 50/50 average of within-primary-cohort supervised and six-indicator peer rank percentiles',
    'consensus_weights':CONSENSUS_WEIGHTS,
    'isolation_forest':'Comparison only; not added to consensus to avoid reusing same feature evidence',
    'labels_used_to_fit_or_select_consensus_weights':False,
    'limitations':[
        'Retrospective ranking; 2023-2024 previously inspected, not independent test',
        'Self-reported diversion is not verified fraud or an unbiased label for undetected fraud',
        'Consensus weights and adverse directions are analyst-defined heuristics, not learned or validated fraud predictors',
        'Correlated financial indicators and overlapping model inputs can still double-count evidence',
        'Peer groups based on revenue only, not mission/subsector',
        'Filing-time availability and amended-filing selection need additional validation',
        'Limited-data filings and Form 990-EZ need separate treatment',
        'Any lead requires independent evidence and human review before investigative action',
    ],
    'seed':SEED,
}
(OUT/'11_manifest.json').write_text(json.dumps(metadata,indent=2))
print('Saved Notebook 11 outputs to:',OUT)

## Interpretation and Next Steps

### 1. Overview

Notebook 11 integrated the supervised machine-learning model developed in Notebook 09 with the financial and governance anomaly-detection framework developed in Notebook 10.

The objective was to determine whether combining these approaches could improve the identification and prioritization of nonprofit organizations exhibiting potentially suspicious financial or governance patterns.

Three primary signals were evaluated:

- **Supervised model:** Identifies patterns associated with organizations that previously self-reported significant diversions of assets.
- **Peer-based anomaly detection:** Identifies organizations exhibiting potentially concerning financial or governance characteristics relative to similarly sized nonprofits.
- **Isolation Forest:** Identifies unusual combinations of financial and governance features without relying on reported diversion labels.

An exploratory combined ranking was developed using equal weights for the supervised and peer-anomaly scores.

The combined ranking is intended to support investigative prioritization, not establish that an organization has committed fraud.

### 2. Dataset and Scoring Population

The analysis included **355,973 Form 990 filings from tax year 2024**.

| Population | Filings |
|---|---:|
| Primary review population | 329,440 |
| Limited-data population | 26,533 |
| **Total** | **355,973** |

The primary review population requires at least four of the six retained peer-anomaly indicators.

The population is smaller than in Notebook 10 because one redundant financial indicator was removed.

Organizations with insufficient information were retained in a separate limited-data population rather than excluded entirely.

All 355,973 filings received supervised model scores.

Final integrity checks found no missing primary consensus scores or duplicate filing identifiers.

### 3. Financial Indicator Correlation and Redundancy

The correlation analysis identified a perfect negative correlation between:

- Expense-to-revenue ratio.
- Net margin.

**Observed correlation: -1.00**

This occurs because net margin is mathematically related to expense-to-revenue ratio when both are calculated using the same revenue denominator.

Including both indicators would effectively count the same financial condition twice in the peer-anomaly score.

**Resolution:** Net margin was removed from the exploratory peer score, reducing the number of indicators from seven to six.

This improves the interpretability of the scoring methodology and reduces one source of redundant weighting.

Additional relationships among the remaining indicators should continue to be monitored.

### 4. Comparison of Supervised and Anomaly-Detection Rankings

The supervised and peer-anomaly approaches identified substantially different organizations.

| Ranking threshold | Organizations appearing in both rankings |
|---|---:|
| Top 100 | 0 |
| Top 500 | 0 |
| Top 1,000 | 32 |

The Spearman rank correlation between the supervised and peer-anomaly rankings was approximately **0.34**.

This indicates a relatively weak-to-moderate positive relationship between the rankings.

**Key finding:** The two approaches are identifying different patterns in the nonprofit population.

The supervised model focuses on characteristics associated with reported asset diversions, while peer-anomaly detection focuses on unusual financial and governance behavior.

These approaches may provide complementary information for investigators.

However, low ranking overlap does not independently demonstrate that either method identifies fraudulent activity.

### 5. Retrospective Comparison Against Reported Asset Diversions

The notebook compared how many organizations with self-reported significant diversions of assets appeared in the highest-ranked groups.

There were **296 positive diversion disclosures** in the primary review population.

| Ranking method | Top 100 | Top 500 | Top 1,000 |
|---|---:|---:|---:|
| Supervised model | 2 | 5 | 8 |
| Peer anomalies (6 indicators) | 0 | 1 | 4 |
| Original peer anomalies (7 indicators) | 0 | 1 | 4 |
| Isolation Forest | 0 | 0 | 0 |
| **Combined ranking** | **2** | **6** | **7** |

The combined ranking identified one additional positive disclosure in the top 500 compared with the supervised model.

However, it identified one fewer positive disclosure in the top 1,000.

Isolation Forest did not identify any reported diversions among its top 1,000 organizations.

**Interpretation:** Combining supervised and anomaly-based signals did not produce a consistent improvement in identifying reported diversion disclosures.

This comparison is informative but limited because reported diversions are not equivalent to confirmed fraud.

An organization may engage in fraudulent activity without disclosing it, while an organization that reports a diversion may have identified and appropriately disclosed misconduct.

Therefore, the absence of reported diversions among highly ranked anomalies does not establish that those anomalies are irrelevant to criminal investigations.

### 6. Combined Ranking and Weight Sensitivity

The exploratory combined ranking uses:

- 50% supervised model ranking.
- 50% peer-anomaly ranking.

The notebook also tested alternative weight combinations.

| Supervised weight | Overlap with 50/50 top 500 |
|---|---:|
| 25% | 403 |
| 50% | 500 |
| 75% | 384 |

Changing the weights caused approximately 97–116 organizations to enter or leave the top 500.

This demonstrates that the combined ranking is sensitive to how the two signals are weighted.

**Conclusion:** Equal weighting is a reasonable exploratory baseline, but it has not been validated as the optimal approach for identifying potential fraud.

The individual model rankings should remain available alongside the combined score so investigators can understand which signals contributed to an organization's prioritization.

### 7. Key Findings

Notebook 11 established several important findings:

1. **The supervised and anomaly-detection approaches identify substantially different organizations.** Their low overlap suggests that they capture different characteristics of the nonprofit population.

2. **Financial indicator redundancy can distort risk rankings.** Removing net margin addressed a confirmed source of double-counting.

3. **Combining signals does not automatically improve performance.** The combined ranking showed mixed results when evaluated against reported asset diversions.

4. **Weighting decisions materially affect investigative priorities.** Different weighting schemes produce meaningfully different top-ranked organizations.

5. **Self-reported diversion is an imperfect measure of potential fraud.** The project should not optimize exclusively for predicting these disclosures.

6. **Explainability is essential.** Investigators need to understand whether an organization was flagged because of historical disclosure-related patterns, unusual financial activity, governance weaknesses, or a combination of signals.

### 8. Remaining Limitations

Several limitations remain:

- **No independently confirmed fraud labels:** The current analysis cannot establish whether top-ranked organizations are actually engaged in fraud.
- **Exploratory scoring weights:** The combined ranking weights have not been validated against investigative outcomes.
- **Peer-group limitations:** Organizations are currently compared primarily by revenue size rather than nonprofit mission or subsector.
- **Limited-data organizations:** Some filings lack sufficient financial and governance information for reliable peer comparisons.
- **Model calibration:** Scores and rankings should not be interpreted as probabilities of fraud.
- **Retrospective analysis:** The results have not been validated for prospective investigative use.
- **Organization-level duplication:** The current output is filing-based and should be reviewed to ensure that organizations do not appear multiple times in the final investigation queue.

### 9. Next Steps: Notebook 12

**Notebook 12 — Investigation Queue and Explainability**

The next phase will focus on transforming the modeling outputs into a practical analytical resource for criminal investigators.

Notebook 12 will:

1. Create an organization-level investigation queue with one prioritized record per EIN.
2. Retain separate supervised, peer-anomaly, Isolation Forest, and combined rankings.
3. Identify organizations flagged by multiple independent approaches.
4. Add financial and governance explanations for elevated anomaly scores.
5. Incorporate historical financial trends and peer-group comparisons.
6. Clearly distinguish organizations with sufficient data from those requiring additional information.
7. Export an analyst-friendly dataset suitable for investigative review.

The investigation queue should provide transparent reasons for prioritization rather than relying on a single opaque fraud-risk score.

### 10. Final Conclusion

Notebook 11 successfully integrated supervised and unsupervised risk indicators and demonstrated that the different approaches identify substantially different nonprofit organizations.

The analysis also confirmed financial indicator redundancy, showed that combined ranking performance is mixed, and established that weighting choices materially influence investigative priorities.

**Decision:** Retain the supervised, peer-anomaly, and Isolation Forest rankings as distinct analytical signals. Preserve the exploratory combined ranking while avoiding claims that its weights or results have been validated for fraud detection.

Proceed to Notebook 12 to develop an explainable, organization-level investigation queue for the criminal investigations team.

The project's ultimate objective remains identifying nonprofit organizations exhibiting potentially suspicious financial and governance patterns that may warrant further investigation—not predicting self-reported diversion disclosures alone.